# 04 · (Optional stretch) Real MLOps tools — MLflow & Evidently

**MLOps Monitoring · Coaching Session · Module 3 (Machine Learning & GenAI)**

---

In notebooks 01–03 we built everything by hand: PSI, a scorecard, a tiny `registry.json`. That is the best way to **understand** it. In a company you would use tools that do the same jobs at scale.

| Job | What we did by hand | Tool | Analogy |
|---|---|---|---|
| Remember every experiment (settings + scores) | printed tables | **MLflow Tracking** | A lab notebook that writes itself |
| Keep every model version + know which one is live | `registry.json` + `set_champion()` | **MLflow Model Registry** | A library catalogue: every edition kept, one marked "on the shelf" |
| Drift reports for many columns | our `psi()` loop | **Evidently** | An automatic health-check report from the clinic |
| Guess performance **before** real prices arrive | — (not covered) | **NannyML** | Guessing exam results from how confident students looked |

**Setup** (run once in your terminal, *not* needed for notebooks 01–03):
```
pip install "mlflow>=3" "evidently>=0.7"
```
⚠️ Evidently changed its code style a lot in 2025 (version 0.7). Many tutorials online use the **old** style (`from evidently.report import Report`) and will not work. This notebook uses the new style.

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

import pandas as pd
from mlops_utils import load_hdb, make_model, score, FEATURES, TARGET, DEPLOY_DATE

df = load_hdb()
reference = df[df.date < DEPLOY_DATE]
latest_month = df.date.max()
holdout = df[df.date > latest_month - pd.DateOffset(months=3)]      # newest 3 months = test
print(f"Holdout (test) months: {holdout.month.min()} to {holdout.month.max()}")

---
## Part A · MLflow Tracking — the self-writing lab notebook 🧪

We repeat the "which window?" question from notebook 03, but let MLflow **record every run**: the settings (*parameters*), the scores (*metrics*) and the model file itself.

In [ ]:
import mlflow
from mlflow import MlflowClient

mlflow.set_tracking_uri("sqlite:///mlflow.db")       # everything is saved in this one file
mlflow.set_experiment("hdb-price-retraining")

# MLflow 3 saves sklearn models with "skops", which only loads types you say you trust.
# We trained these models ourselves, so we trust the tree type used by HistGradientBoosting.
TRUSTED = ["sklearn.ensemble._hist_gradient_boosting.predictor.TreePredictor"]

def log_model(model, example):
    try:
        return mlflow.sklearn.log_model(model, name="model", registered_model_name="hdb_price",
                                        input_example=example, skops_trusted_types=TRUSTED)
    except TypeError:                                  # older MLflow without skops options
        return mlflow.sklearn.log_model(model, name="model", registered_model_name="hdb_price",
                                        input_example=example)

for window_years in [1, 2, 4]:
    train = df[(df.date <= latest_month - pd.DateOffset(months=3)) &
               (df.date >  latest_month - pd.DateOffset(months=3, years=window_years))]
    with mlflow.start_run(run_name=f"sliding_{window_years}y"):
        model = make_model().fit(train[FEATURES], train[TARGET])
        s = score(holdout[TARGET], model.predict(holdout[FEATURES]))
        # 🔧 YOUR TURN: record the window size as a parameter
        mlflow.log_param("window_years", ____)
        mlflow.log_param("train_rows", len(train))
        mlflow.log_metrics({"MAPE": s["MAPE_%"], "MAE": s["MAE"], "bias": s["bias"]})
        info = log_model(model, train[FEATURES].head(3))
        print(f"window {window_years}y -> MAPE {s['MAPE_%']:.2f}%  (registered as version {info.registered_model_version})")

<details><summary>▸ Hint</summary>

The loop variable holding the window size is `window_years`.
</details>

### See it in the MLflow web page
In a terminal, inside this `notebooks` folder, run:
```
mlflow ui --backend-store-uri sqlite:///mlflow.db
```
Open <http://127.0.0.1:5000>. Click the experiment, tick the three runs and press **Compare**.

In [ ]:
runs = mlflow.search_runs(experiment_names=["hdb-price-retraining"])
runs[["tags.mlflow.runName", "params.window_years", "metrics.MAPE", "metrics.bias"]].sort_values("metrics.MAPE")

---
## Part B · MLflow Model Registry — champion alias and one-line rollback 🏷️

The registry keeps **every version**. An **alias** such as `champion` points at the one that is live. Your app always loads `models:/hdb_price@champion`, so it never needs to change.

Rolling back = **moving the alias** to an older version. Same idea as `set_champion()` in notebook 03.

In [ ]:
client = MlflowClient()
versions = client.search_model_versions("name='hdb_price'")
mape_of = {v.version: client.get_run(v.run_id).data.metrics["MAPE"] for v in versions}
best = min(mape_of, key=mape_of.get)

client.set_registered_model_alias("hdb_price", "champion", best)
print(f"Champion is now version {best} (MAPE {mape_of[best]:.2f}%)")

live = mlflow.sklearn.load_model("models:/hdb_price@champion")
print("The app loads 'models:/hdb_price@champion' ->", round(live.predict(holdout[FEATURES].head(1))[0]), "for the first flat")

In [ ]:
# Something is wrong with the champion -> roll back to another version
# 🔧 YOUR TURN: pick a different version number to point the alias at
rollback_to = ____
client.set_registered_model_alias("hdb_price", "champion", rollback_to)
print("Champion alias now points to version", client.get_model_version_by_alias("hdb_price", "champion").version)

<details><summary>▸ Hint</summary>

Any version number other than `best` works, e.g. `"1"`. Version numbers are strings.
</details>

---
## Part C · Evidently — an automatic drift report 🩺

Evidently compares reference vs current data for **every column** and creates an HTML report you can open in a browser or send to your team.

In [ ]:
from evidently import Report
from evidently.presets import DataDriftPreset

current = df[df.date.dt.year == 2025]
report = Report([DataDriftPreset()])
snapshot = report.run(reference_data=reference[FEATURES].sample(5000, random_state=0),
                      current_data=current[FEATURES].sample(5000, random_state=0))
snapshot.save_html("drift_report_2025.html")
print("Saved drift_report_2025.html — open it in your browser")

for m in snapshot.dict()["metrics"]:
    name = m.get("metric_name", m.get("metric_id", ""))
    if name.startswith("ValueDrift"):
        print(f"{name:<95} score = {m['value']:.3f}")

**Look closely:** Evidently may call `floor_area_sqm` **drifted**, while our PSI said 🟢 stable (about 0.03).

Both are "right". **They measure different things with different default thresholds.** For numbers, Evidently's default here is the *Wasserstein distance* with a 0.1 cut-off, not PSI.

✅ **Lesson:** never trust a red/green light without knowing **which test** and **which threshold** produced it. You can change both in Evidently, e.g. `DataDriftPreset(method="psi")`.

---
## Part D · NannyML — reading only (no code) 📖

Sometimes real outcomes arrive **very late**, e.g. a loan default may take 2 years to show up. NannyML tries to **estimate** the model's error *before* the real answers arrive (for regression it uses a method called **DLE**: a second model learns to predict the first model's errors).

> **Analogy:** a teacher guessing the class's exam results from how unsure the students looked while writing, before marking any paper.

**For HDB we don't really need it:** real prices arrive within weeks, so we can simply measure the error (notebook 02). It also has heavy dependencies that often clash with other packages. Worth knowing the idea; try it only in a fresh environment.

---
### What we skipped on purpose (out of scope today)
Serving the model as an API (FastAPI), containers (Docker), scheduled pipelines (Airflow / cron), cloud platforms, and alerting dashboards (Grafana). They **automate** what you did today; the *thinking* stays the same.